In [1]:
# ── Cell 1 — Imports ───────────────────────────────────────────────────────
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from pathlib import Path

Path('../data/merged').mkdir(parents=True, exist_ok=True)

In [4]:
# ── Cell 2 — Load both datasets ────────────────────────────────────────────
pbdb = pd.read_csv('../data/pbdb/icewave_occurrences.csv')
idib = pd.read_csv('../data/idigbio/idigbio_occurrences.csv')

# Standardize columns
pbdb['source'] = 'pbdb'
pbdb['taxon'] = pbdb['taxon_name'].str.split().str[0]  # genus only
pbdb = pbdb[['taxon','latitude','longitude','source']].copy()

idib = idib[['taxon','latitude','longitude','source']].copy()

print(f'PBDB:    {len(pbdb)} records')
print(f'iDigBio: {len(idib)} records')

combined = pd.concat([pbdb, idib], ignore_index=True)
print(f'Combined before dedup: {len(combined)}')

PBDB:    78 records
iDigBio: 304 records
Combined before dedup: 382


In [5]:
# ── Cell 3 — Deduplicate (round to 4 decimal places ~11m) ─────────────────
combined['lat_r'] = combined['latitude'].round(4)
combined['lon_r'] = combined['longitude'].round(4)
combined = combined.drop_duplicates(subset=['taxon','lat_r','lon_r'])
combined = combined.drop(columns=['lat_r','lon_r'])
combined = combined.reset_index(drop=True)

print(f'After dedup: {len(combined)} unique occurrences')
print(combined.groupby(['taxon','source']).size().unstack(fill_value=0).to_string())

After dedup: 133 unique occurrences
source       idigbio  pbdb
taxon                     
Arctodus           2     3
Bison             10    10
Camelops           0     7
Cervus            28     3
Equus              4    16
Mammut            11     5
Mammuthus          7    16
Paramylodon        3     8


In [7]:
# ── Cell 4 — Add lithology from SGMC ──────────────────────────────────────
print('Loading SGMC geology...')
sgmc = gpd.read_file('../data/geology/SGMC/USGS_SGMC_Shapefiles/SGMC_Geology.shp')
three_states = sgmc[sgmc['STATE'].isin(['WA','OR','NV'])].copy()

# Build occurrence GeoDataFrame
gdf = gpd.GeoDataFrame(
    combined,
    geometry=[Point(r.longitude, r.latitude) for _, r in combined.iterrows()],
    crs='EPSG:4326'
)

# Reproject SGMC to match occurrences
three_states = three_states.to_crs('EPSG:4326')

# Spatial join
print('Joining lithology to occurrences...')
joined = gpd.sjoin(gdf, three_states[['MAJOR1','MAJOR2','AGE_MIN','geometry']],
                   how='left', predicate='within')

combined['lithology'] = joined['MAJOR1'].values
combined['lith2']     = joined['MAJOR2'].values
combined['geo_age']   = joined['AGE_MIN'].values

print(f'\nTop lithology types at fossil sites:')
print(combined['lithology'].value_counts().head(10).to_string())

Loading SGMC geology...
Joining lithology to occurrences...

Top lithology types at fossil sites:
lithology
Basalt             27
Clay               21
Andesite           11
Silt                7
Sandstone           6
Coarse-detrital     5
Sand                4
Rhyolite            4
Graywacke           4
Metasedimentary     3


In [10]:
# ── Cell 5 — Encode lithology as numeric feature ──────────────────────────
LITH_SCORE = {
    # Excellent — unconsolidated sediments, perfect preservation
    'Clay':              1.0,
    'Silt':              1.0,
    'Sand':              1.0,
    'Gravel':            1.0,
    'Unconsolidated':    1.0,
    'Alluvium':          1.0,
    'Loess':             1.0,
    # Good — sedimentary rock
    'Sandstone':         0.9,
    'Mudstone':          0.9,
    'Shale':             0.9,
    'Limestone':         0.9,
    'Carbonate':         0.9,
    'Coarse-detrital':   0.85,
    'Conglomerate':      0.8,
    'Evaporite':         0.8,
    'Graywacke':         0.7,
    'Sedimentary':       0.9,
    # Poor — volcanic and igneous
    'Basalt':            0.15,
    'Andesite':          0.15,
    'Rhyolite':          0.15,
    'Volcanic':          0.15,
    'Igneous, intrusive':0.1,
    'Tuff':              0.3,
    # Very poor
    'Metamorphic':       0.1,
    'Metasedimentary':   0.2,
    'Granitic':          0.1,
}
combined['lith_score'] = combined['lithology'].map(LITH_SCORE).fillna(0.4)

print('Lithology score distribution:')
print(combined['lith_score'].value_counts().sort_index(ascending=False).to_string())
print(f'\nMean lith score at fossil sites: {combined["lith_score"].mean():.3f}')
print(f'Records with high lith score (>0.8): {(combined["lith_score"] > 0.8).sum()}')

Lithology score distribution:
lith_score
1.00    33
0.90    11
0.85     5
0.80     2
0.70     4
0.40    33
0.20     3
0.15    42

Mean lith score at fossil sites: 0.539
Records with high lith score (>0.8): 49


In [11]:
# ── Cell 6 — Save merged dataset ──────────────────────────────────────────
combined.to_csv('../data/merged/icewave_merged_occurrences.csv', index=False)

gdf_out = gpd.GeoDataFrame(
    combined,
    geometry=[Point(r.longitude, r.latitude) for _, r in combined.iterrows()],
    crs='EPSG:4326'
)
gdf_out.to_file('../data/merged/icewave_merged_occurrences.geojson', driver='GeoJSON')

print(f'Saved {len(combined)} merged occurrences')
print(f'\nBy taxon:')
print(combined['taxon'].value_counts().to_string())
print(f'\nBy source:')
print(combined['source'].value_counts().to_string())

Saved 133 merged occurrences

By taxon:
taxon
Cervus         31
Mammuthus      23
Equus          20
Bison          20
Mammut         16
Paramylodon    11
Camelops        7
Arctodus        5

By source:
source
pbdb       68
idigbio    65


In [18]:
# ── Cell 7 — Fetch paleolake/playa polygons via USGS NHD REST API ─────────
import requests, json
import geopandas as gpd
from shapely.geometry import shape
from pathlib import Path

Path('../data/paleolakes').mkdir(parents=True, exist_ok=True)

# NHD REST endpoint — layer 9 is NHDWaterbody
# FType 361=Playa, 390=LakePond
BASE = 'https://hydro.nationalmap.gov/arcgis/rest/services/nhd/MapServer/12/query'

all_features = []
for ftype, label in [(361, 'Playa'), (390, 'LakePond')]:
    print(f'Fetching {label} (FTYPE={ftype})...')
    params = {
        'where':             f'FTYPE={ftype}',
        'geometry':          '-125.0,35.5,-114.0,49.5',
        'geometryType':      'esriGeometryEnvelope',
        'inSR':              '4326',
        'spatialRel':        'esriSpatialRelIntersects',
        'outFields':         'FTYPE,GNIS_NAME,AREASQKM',
        'returnGeometry':    'true',
        'outSR':             '4326',
        'f':                 'geojson',
        'resultRecordCount': 2000,
    }
    try:
        r = requests.get(BASE, params=params, timeout=120)
        if r.status_code == 200:
            data = r.json()
            feats = data.get('features', [])
            print(f'  Got {len(feats)} features')
            all_features.extend(feats)
        else:
            print(f'  HTTP {r.status_code}: {r.text[:200]}')
    except Exception as e:
        print(f'  ERROR: {e}')

if all_features:
    gdf = gpd.GeoDataFrame.from_features(all_features, crs='EPSG:4326')
    if 'AREASQKM' in gdf.columns:
        gdf = gdf[gdf['AREASQKM'] >= 0.1]
    print(f'\nTotal waterbodies after size filter: {len(gdf)}')
    print(gdf['FTYPE'].value_counts().to_string())
    gdf.to_file('../data/paleolakes/nhd_waterbodies.geojson', driver='GeoJSON')
    print('Saved: data/paleolakes/nhd_waterbodies.geojson')
else:
    print('No features returned')

Fetching Playa (FTYPE=361)...
  Got 2000 features
Fetching LakePond (FTYPE=390)...
  Got 2000 features

Total waterbodies after size filter: 202
FTYPE
361    163
390     39
Saved: data/paleolakes/nhd_waterbodies.geojson


In [21]:
import geopandas as gpd
import numpy as np
from shapely.geometry import Point

print('Loading waterbodies...')
water = gpd.read_file('../data/paleolakes/nhd_waterbodies.geojson')

gdf = gpd.GeoDataFrame(
    combined,
    geometry=[Point(r.longitude, r.latitude) for _, r in combined.iterrows()],
    crs='EPSG:4326'
).to_crs('EPSG:32610')

water_utm = water.to_crs('EPSG:32610')
water_union = water_utm.union_all()

print('Computing distances...')
combined['dist_to_water_m'] = gdf.geometry.apply(
    lambda g: g.distance(water_union)
)

print(combined['dist_to_water_m'].describe().round(1).to_string())
print(f'\nWithin 5km:  {(combined["dist_to_water_m"] < 5000).sum()} / {len(combined)}')
print(f'Within 10km: {(combined["dist_to_water_m"] < 10000).sum()} / {len(combined)}')

Loading waterbodies...
Computing distances...
count       133.0
mean     108048.2
std       60245.0
min        5006.3
25%       64410.8
50%      109335.7
75%      161809.8
max      222715.5

Within 5km:  0 / 133
Within 10km: 12 / 133


In [25]:
print(combined['dist_to_water_m'].describe().round(1).to_string())
print(f'\nWithin 5km:  {(combined["dist_to_water_m"] < 5000).sum()} / {len(combined)}')
print(f'Within 10km: {(combined["dist_to_water_m"] < 10000).sum()} / {len(combined)}')
print(f'Within 25km: {(combined["dist_to_water_m"] < 25000).sum()} / {len(combined)}')

count       133.0
mean     126660.5
std       99174.5
min        3119.3
25%       32738.4
50%      153101.5
75%      170621.6
max      367947.4

Within 5km:  13 / 133
Within 10km: 18 / 133
Within 25km: 30 / 133


In [24]:
# ── Cell 8b — Waterbodies with proper state filter ─────────────────────────
BASE = 'https://hydro.nationalmap.gov/arcgis/rest/services/nhd/MapServer/12/query'
all_features = []

# Tighter bbox — just OR/NV/WA, and add minimum size filter in the WHERE clause
# Also limit LakePond to larger lakes only (>=1 km2) to keep it manageable
QUERIES = [
    (361, 'Playa',    'FTYPE=361 AND AREASQKM>=0.01'),
    (390, 'LakePond', 'FTYPE=390 AND AREASQKM>=1.0'),  # only lakes >1km2
]

for ftype, label, where in QUERIES:
    offset = 0
    while True:
        params = {
            'where':             where,
            'geometry':          '-125.0,35.5,-114.0,49.5',
            'geometryType':      'esriGeometryEnvelope',
            'inSR':              '4326',
            'spatialRel':        'esriSpatialRelIntersects',
            'outFields':         'FTYPE,GNIS_NAME,AREASQKM',
            'returnGeometry':    'true',
            'outSR':             '4326',
            'f':                 'geojson',
            'resultRecordCount': 2000,
            'resultOffset':      offset,
        }
        r = requests.get(BASE, params=params, timeout=120)
        if r.status_code != 200:
            break
        feats = r.json().get('features', [])
        if not feats:
            break
        all_features.extend(feats)
        print(f'  {label} offset {offset}: {len(feats)} features')
        if len(feats) < 2000:
            break
        offset += 2000

print(f'\nTotal raw features: {len(all_features)}')
if all_features:
    gdf_w = gpd.GeoDataFrame.from_features(all_features, crs='EPSG:4326')
    print(f'Features: {len(gdf_w)}')
    print(gdf_w['FTYPE'].value_counts().to_string())
    gdf_w.to_file('../data/paleolakes/nhd_waterbodies.geojson', driver='GeoJSON')
    print('Saved.')

    water_utm = gdf_w.to_crs('EPSG:32610')
    water_union = water_utm.union_all()
    combined['dist_to_water_m'] = gdf.geometry.apply(
        lambda g: g.distance(water_union)
    )
    print(f'\nWithin 5km:  {(combined["dist_to_water_m"] < 5000).sum()} / {len(combined)}')
    print(f'Within 10km: {(combined["dist_to_water_m"] < 10000).sum()} / {len(combined)}')
    print(combined['dist_to_water_m'].describe().round(1).to_string())

  Playa offset 0: 2000 features
  Playa offset 2000: 1980 features

Total raw features: 3980
Features: 3980
FTYPE
361    3980
Saved.

Within 5km:  13 / 133
Within 10km: 18 / 133
count       133.0
mean     126660.5
std       99174.5
min        3119.3
25%       32738.4
50%      153101.5
75%      170621.6
max      367947.4


In [22]:
combined['dist_to_water_m'] = gdf.geometry.apply(
    lambda g: g.distance(water_union)
)
print(combined['dist_to_water_m'].describe().round(1).to_string())
print(f'\nWithin 5km:  {(combined["dist_to_water_m"] < 5000).sum()} / {len(combined)}')
print(f'Within 10km: {(combined["dist_to_water_m"] < 10000).sum()} / {len(combined)}')

count       133.0
mean     108048.2
std       60245.0
min        5006.3
25%       64410.8
50%      109335.7
75%      161809.8
max      222715.5

Within 5km:  0 / 133
Within 10km: 12 / 133


In [26]:
# ── Cell 9 — Save final enriched dataset ──────────────────────────────────
combined.to_csv('../data/merged/icewave_merged_occurrences.csv', index=False)

gdf_out = gpd.GeoDataFrame(
    combined,
    geometry=[Point(r.longitude, r.latitude) for _, r in combined.iterrows()],
    crs='EPSG:4326'
)
gdf_out.to_file('../data/merged/icewave_merged_occurrences.geojson', driver='GeoJSON')

print(f'Saved {len(combined)} enriched occurrences')
print(f'\nBy taxon:')
print(combined['taxon'].value_counts().to_string())
print(f'\nFeatures available for model v2:')
print('  elevation, slope, aspect, tri  — terrain (existing)')
print('  lith_score                      — NEW: lithology quality')
print('  dist_to_water_m                 — available as bonus')

Saved 133 enriched occurrences

By taxon:
taxon
Cervus         31
Mammuthus      23
Equus          20
Bison          20
Mammut         16
Paramylodon    11
Camelops        7
Arctodus        5

Features available for model v2:
  elevation, slope, aspect, tri  — terrain (existing)
  lith_score                      — NEW: lithology quality
  dist_to_water_m                 — available as bonus
